# Fine-tune T5-small for Grammar Error Correction
This notebook follows the supplied project report: 5,000 C4_200M pairs, `grammar:` prefix, max length 64, one epoch, AdamW, and a saved `gec_t5` model.

**Runtime:** Google Colab → Runtime → Change runtime type → T4 GPU (or another available GPU).

Training is optional for running the local demo. Run this notebook when you want your own fine-tuned checkpoint.

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate nltk
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import T5ForConditionalGeneration, T5TokenizerFast, get_linear_schedule_with_warmup
from pathlib import Path
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# Stream the dataset and retain the first 5,000 sentence pairs.
ds = load_dataset('liweili/c4_200m', split='train', streaming=True)
pairs = [(r['input'], r['output']) for r in ds.take(5000)]
pairs = [(a, b) for a, b in pairs if isinstance(a, str) and isinstance(b, str) and a.strip() and b.strip()]
print('Training pairs:', len(pairs))
print('Example:', pairs[0])

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = T5TokenizerFast.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small').to(device)

def collate(batch):
    inputs = tokenizer(['grammar: ' + a for a, _ in batch], max_length=64, truncation=True, padding=True, return_tensors='pt')
    labels = tokenizer([b for _, b in batch], max_length=64, truncation=True, padding=True, return_tensors='pt').input_ids
    labels[labels == tokenizer.pad_token_id] = -100
    return inputs, labels

loader = DataLoader(pairs, batch_size=32, shuffle=True, collate_fn=collate)
epochs = 1
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
total_steps = epochs * len(loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05 * total_steps), total_steps)
print('Batches:', len(loader), '| Device:', device)

In [ ]:
model.train()
for epoch in range(epochs):
    for step, (inputs, labels) in enumerate(loader):
        inputs = {k: v.to(device) for k, v in inputs.items()}
        labels = labels.to(device)
        loss = model(**inputs, labels=labels).loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        if step % 20 == 0:
            print(f'epoch {epoch} step {step}/{len(loader)} loss {loss.item():.4f}')

output_dir = 'gec_t5'
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print('Saved model to', output_dir)

In [ ]:
# Quick smoke test of the fine-tuned model.
model.eval()
def correct(sentence):
    encoded = tokenizer('grammar: ' + sentence, return_tensors='pt', max_length=64, truncation=True).to(device)
    with torch.inference_mode():
        generated = model.generate(**encoded, max_length=64, num_beams=4)
    return tokenizer.decode(generated[0], skip_special_tokens=True)

examples = [
    'She go to school yesterday and buyed a book.',
    "He don't like apples.",
    'I has a important meeting at tomorrow.'
]
for sentence in examples:
    print('INPUT :', sentence)
    print('OUTPUT:', correct(sentence))
    print()

## Download the fine-tuned model
Download the `gec_t5` folder from the Colab Files panel (right-click → Download). Extract it into your local project at `models/gec_t5/`. The local Gradio app will then load your fine-tuned checkpoint automatically.